In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import os

# Set gaya visualisasi agar terlihat profesional
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# ==============================================================================
# 1. INSISIALISASI & PEMBACAAN DATA
# ==============================================================================
input_path = "data/3_Data_Normalized/Hasil_ZScore_Data_Normalization.xlsx"
folder_output = "data/4_Data_Clusterized/"

print("Membaca data dari:", input_path)
df = pd.read_excel(input_path)

# Memisahkan fitur untuk clustering (9 Variabel)
X = df.drop(columns=['Program Studi'])

print("-" * 60)

# ==============================================================================
# 2. METODE ELBOW (VISUALISASI 1)
# ==============================================================================
inertia = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

# Plotting Grafik Elbow
plt.figure(figsize=(9, 5))
plt.plot(K_range, inertia, marker='o', linestyle='--', color='#1f77b4', linewidth=2, markersize=8)
plt.title('Metode Elbow untuk Menentukan Jumlah Klaster Optimal (9 Variabel QS)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Jumlah Klaster (K)', fontsize=10)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=10)
plt.xticks(K_range)
plt.grid(True, linestyle=':', alpha=0.6)

# Menyimpan grafik elbow
plt.savefig(f"{folder_output}1_Grafik_Elbow_Method.png", dpi=300, bbox_inches='tight')
print("Visualisasi 1: Grafik Elbow berhasil ditampilkan dan disimpan.")
plt.show()

# ==============================================================================
# 3. K-MEANS 9 VARIABEL & VISUALISASI CLUSTER 2D VIA PCA (VISUALISASI 2)
# ==============================================================================
# Tentukan K optimal berdasarkan grafik elbow (misal K = 3)
k_optimal = 3 
kmeans_final = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df['Cluster_Keseluruhan'] = kmeans_final.fit_predict(X)

# Reduksi dimensi dari 9 variabel menjadi 2 komponen menggunakan PCA agar bisa diplot ke grafik 2D
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

# Plotting Scatter Plot Hasil Klasterisasi Keseluruhan
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='PCA1', y='PCA2', 
    hue='Cluster_Keseluruhan', 
    palette='Set1', 
    data=df, 
    s=100, alpha=0.8, edgecolor='w'
)
plt.title(f'Visualisasi Klasterisasi Program Studi (K={k_optimal}) Berdasarkan 9 Variabel QS Asia', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Principal Component 1 (PC1)', fontsize=10)
plt.ylabel('Principal Component 2 (PC2)', fontsize=10)
plt.legend(title='Klaster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)

# Menyimpan grafik klaster keseluruhan
plt.savefig(f"{folder_output}2_Visualisasi_Cluster_Keseluruhan.png", dpi=300, bbox_inches='tight')
print("Visualisasi 2: Scatter Plot Klaster Keseluruhan (9 Var) berhasil ditampilkan dan disimpan.")
plt.show()

# Simpan data tabular klaster keseluruhan (Hapus kolom PCA sementara agar file bersih)
df_save_keseluruhan = df.drop(columns=['PCA1', 'PCA2'])
df_save_keseluruhan.to_excel(f"{folder_output}Hasil_Klasterisasi_Keseluruhan.xlsx", index=False)

print("-" * 60)

# ==============================================================================
# 4. K-MEANS 1 VARIABEL (PPF) & VISUALISASI TIERING (VISUALISASI 3)
# ==============================================================================
X_ppf = df[['PPF']]
k_ppf = 3

kmeans_ppf = KMeans(n_clusters=k_ppf, random_state=42, n_init=10)
df_ppf = df[['Program Studi', 'PPF']].copy()
df_ppf['Cluster_PPF'] = kmeans_ppf.fit_predict(X_ppf)

# Membuat indeks data buatan pada sumbu X untuk merentangkan titik data prodi agar tidak menumpuk vertikal
df_ppf['Prodi_Index'] = range(len(df_ppf))

# Plotting Scatter Plot untuk Klasterisasi 1D PPF
plt.figure(figsize=(11, 6))
sns.scatterplot(
    x='Prodi_Index', y='PPF', 
    hue='Cluster_PPF', 
    palette='viridis', 
    data=df_ppf, 
    s=80, alpha=0.9
)
plt.title(f'Klasterisasi 1 Dimensi: Pengelompokan Prodi Berdasarkan Produktivitas Riset (PPF)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Indeks Program Studi (Distribusi Data)', fontsize=10)
plt.ylabel('Nilai Z-Score Papers per Faculty (PPF)', fontsize=10)
plt.legend(title='Klaster PPF', loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)

# Menyimpan grafik klaster PPF
plt.savefig(f"{folder_output}3_Visualisasi_Cluster_PPF.png", dpi=300, bbox_inches='tight')
print("Visualisasi 3: Scatter Plot Klasterisasi 1D PPF berhasil ditampilkan dan disimpan.")
plt.show()

# Simpan data tabular klaster PPF
df_ppf.drop(columns=['Prodi_Index']).to_excel(f"{folder_output}Hasil_Klasterisasi_PPF.xlsx", index=False)
print("\n[PROSES SELESAI]: Cek folder 'data/4_Data_Clusterized/' untuk melihat file Excel dan gambar grafik!")

In [ ]:
# Load data
df = pd.read_csv("data/3_Data_Normalized/Hasil_ZScore_Data_Normalization.csv")  # atau .xlsx

# ============================================================
# FUNGSI DETEKSI ELBOW OTOMATIS
# ============================================================

def find_elbow_perpendicular(k_values, inertia_values):
    """Deteksi elbow via jarak tegak lurus ke garis referensi"""
    k_arr = np.array(k_values, dtype=float)
    i_arr = np.array(inertia_values, dtype=float)

    k_norm = (k_arr - k_arr.min()) / (k_arr.max() - k_arr.min())
    i_norm = (i_arr - i_arr.min()) / (i_arr.max() - i_arr.min())

    p1 = np.array([k_norm[0], i_norm[0]])
    p2 = np.array([k_norm[-1], i_norm[-1]])

    distances = []
    for i in range(len(k_norm)):
        p = np.array([k_norm[i], i_norm[i]])
        d = np.abs(np.cross(p2 - p1, p1 - p)) / np.linalg.norm(p2 - p1)
        distances.append(d)

    return k_values[np.argmax(distances)], distances


def find_elbow_second_derivative(k_values, inertia_values):
    """Deteksi elbow via turunan kedua inertia"""
    i_arr = np.array(inertia_values, dtype=float)
    second_diff = np.diff(i_arr, 2)
    optimal_idx = np.argmax(second_diff) + 1  # shift karena diff mengurangi panjang array
    return k_values[optimal_idx]


# ============================================================
# JALANKAN K-MEANS + SILHOUETTE SCORE SEKALIGUS
# ============================================================
inertia       = []
sil_scores    = []
K_range       = list(range(1, 11))
K_range_sil   = list(range(2, 11))

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X)
    inertia.append(km.inertia_)
    if k >= 2:
        sil_scores.append(silhouette_score(X, labels))


# ============================================================
# DETEKSI K OPTIMAL (3 METODE)
# ============================================================
k_opt_perp, distances = find_elbow_perpendicular(K_range, inertia)
k_opt_2nd             = find_elbow_second_derivative(K_range, inertia)
k_opt_sil             = K_range_sil[np.argmax(sil_scores)]


# ============================================================
# VISUALISASI 3 PANEL
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Elbow Curve ---
ax1 = axes[0]
ax1.plot(K_range, inertia, 'o-', color='steelblue', linewidth=2, markersize=6)
ax1.axvline(x=k_opt_perp, color='red',    linestyle='--', linewidth=1.5, label=f'Perp. Distance : K={k_opt_perp}')
ax1.axvline(x=k_opt_2nd,  color='orange', linestyle='--', linewidth=1.5, label=f'2nd Derivative : K={k_opt_2nd}')
ax1.set_title('Elbow Method', fontsize=12, fontweight='bold')
ax1.set_xlabel('Jumlah Cluster (K)')
ax1.set_ylabel('Inertia')
ax1.legend(fontsize=9)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.set_xticks(K_range)

# --- Panel 2: Perpendicular Distance ---
ax2 = axes[1]
ax2.plot(K_range, distances, 's-', color='green', linewidth=2, markersize=6)
ax2.scatter(k_opt_perp, distances[K_range.index(k_opt_perp)],
            color='red', s=120, zorder=5, label=f'Elbow: K={k_opt_perp}')
ax2.set_title('Perpendicular Distance\n(Knee Detection)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Jumlah Cluster (K)')
ax2.set_ylabel('Jarak Tegak Lurus (Normalized)')
ax2.legend(fontsize=9)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.set_xticks(K_range)

# --- Panel 3: Silhouette Score ---
ax3 = axes[2]
ax3.plot(K_range_sil, sil_scores, '^-', color='purple', linewidth=2, markersize=6)
ax3.scatter(k_opt_sil, max(sil_scores),
            color='red', s=120, zorder=5, label=f'Max Score: K={k_opt_sil}')
ax3.set_title('Silhouette Score\n(Validasi Kualitas Cluster)', fontsize=12, fontweight='bold')
ax3.set_xlabel('Jumlah Cluster (K)')
ax3.set_ylabel('Silhouette Score')
ax3.legend(fontsize=9)
ax3.grid(True, linestyle='--', alpha=0.6)
ax3.set_xticks(K_range_sil)

plt.suptitle('Analisis K Optimal — K-Means Clustering', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('k_optimal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


# ============================================================
# REKOMENDASI FINAL (VOTING SYSTEM)
# ============================================================
votes      = [k_opt_perp, k_opt_2nd, k_opt_sil]
vote_count = Counter(votes)
k_majority, n_agree = vote_count.most_common(1)[0]

print("=" * 55)
print("         HASIL ANALISIS K OPTIMAL")
print("=" * 55)
print(f"  Metode 1 - Perpendicular Distance : K = {k_opt_perp}")
print(f"  Metode 2 - Second Derivative      : K = {k_opt_2nd}")
print(f"  Metode 3 - Silhouette Score       : K = {k_opt_sil}  (skor: {max(sil_scores):.4f})")
print("-" * 55)

if n_agree >= 2:
    print(f"\n  ✅ REKOMENDASI FINAL  →  K = {k_majority}")
    print(f"     ({n_agree} dari 3 metode sepakat)")
else:
    print(f"\n  ⚠️  Tidak ada konsensus antar metode.")
    print(f"     Rekomendasi utama (Silhouette) → K = {k_opt_sil}")
    print(f"     Pertimbangkan juga             → K = {k_opt_perp}")

print("\n  Silhouette Score per K:")
for k, s in zip(K_range_sil, sil_scores):
    marker = "  ← terbaik" if k == k_opt_sil else ""
    print(f"    K={k:2d} : {s:.4f}{marker}")
print("=" * 55)

In [ ]:
# ── 1. Persiapkan Fitur Numerik ───────────────────────────────
# Mengambil array numpy dari dataframe yang sudah dinormalisasi
X = df_normalized[kolom_qs].values

# ── 2. Hitung WCSS untuk K = 1 sampai 10 ──────────────────────
wcss = []
max_k = 10

print("=== DETAIL NILAI WCSS (INERTIA) UNTUK SETIAP NILAI K ===")
print("-" * 50)
for k in range(1, max_k + 1):
    # Menggunakan n_init=10 untuk stabilitas hasil dan random_state=42 agar konsisten
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)
    print(f"Jumlah Klaster (K = {k:2d}) -> Nilai WCSS: {kmeans.inertia_:.4f}")
print("-" * 50)

# ── 3. Visualisasi Grafik Elbow Method ──────────────────────────
plt.figure(figsize=(10, 6))

# Membuat garis plot utama
plt.plot(range(1, max_k + 1), wcss, marker='o', linestyle='-', color='#2c3e50', 
         markerfacecolor='#e74c3c', markersize=8, linewidth=2, label='WCSS Value')

# Menambahkan anotasi angka nilai WCSS tepat di atas setiap titik marker
for i, nilai_wcss in enumerate(wcss):
    plt.annotate(f"{nilai_wcss:.1f}", 
                 (i + 1, wcss[i]), 
                 textcoords="offset points", 
                 xytext=(0, 10), 
                 ha='center', 
                 fontsize=9, 
                 fontweight='bold', 
                 color='#2980b9')

# Aturan estetika grafik
plt.title('Elbow Method: Mencari Jumlah Klaster Optimal', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Jumlah Klaster (K)', fontsize=12)
plt.ylabel('Nilai WCSS (Inertia)', fontsize=12)
plt.xticks(range(1, max_k + 1))
plt.grid(True, linestyle='--', alpha=0.5)

# Memberikan sedikit ruang vertikal di atas grafik agar teks anotasi tidak terpotong
plt.ylim(min(wcss) - (max(wcss)*0.05), max(wcss) + (max(wcss)*0.08))

plt.tight_layout()
plt.show()